In [ ]:
#install.packages("ggpubr")
library(ggplot2)
library(behavr)
library(scopr)
library(sleepr)
library(ggetho)
library(plotly)
#library(survival)
library(cowplot)
#library(ggthemes)
library(plotly)
library(data.table)
library(stringi)
library(ggtern)
library(ggpubr)
library(EnvStats)
library(RColorBrewer)
library(dplyr)
library(plyr)
library(survival)
library(survminer)

In [ ]:
####DEFINE FOLDERS AND DOCUMENTS/METADATA####
REMOTE_DATA_SOURCE <- "ftp://turing.lab.gilest.ro/auto_generated_data/ethoscope_results/"
MY_DIR <- "/home/hjones/insecticide_assay/survival/"
setwd(MY_DIR)
DATA_DIR <- "/mnt/ethoscope_results"
CACHE <- "/home/cache"
METADATA <- "/home/hjones/insecticide_assay/survival/20210922_ethoscope_survival_insecticides_1000.csv"

In [ ]:
#To get the files from the remote source
query <- link_ethoscope_metadata(METADATA,result_dir = DATA_DIR)

In [ ]:
dt <- load_ethoscope(query,
                     reference_hour = 9.0, 
                     FUN = sleep_annotation,
                     velocity_correction_coef = 0.01,
                     cache = CACHE)

In [ ]:
dt[,t:=t+days(xmv(baseline_days))]

In [ ]:
dt <- dt[t >days(0) & t< days(1)]

In [ ]:
#This calculates the hours between t==0 (insecticide exposure day 9am) to the last movement in second.
lifespan_dt <- dt[xmv(concentration)=="1000"][t>0,.(lifespan = (max(t) - (min(t)))/hours(1)), by=id]
lifespan_dt <- rejoin(lifespan_dt)

In [ ]:
lifespan_dt <- apply(lifespan_dt,2,as.character)

In [ ]:
write.table(lifespan_dt, 
            "/home/hjones/insecticide_assay/survival/lifespan_summary.txt", 
            sep="\t", 
            row.names=FALSE)

In [ ]:
lifespan_processed = fread("/home/hjones/insecticide_assay/survival/lifespan_summary.txt",
                           header = TRUE)
print(lifespan_processed)

In [ ]:
#convert lifespan metric into numeric
lifespan_processed$lifespan <- as.numeric(as.character(lifespan_processed$lifespan))

In [ ]:
#to make a basic survival curve
iwillsurvive_insecticides <-survfit(Surv(lifespan_processed$lifespan)~lifespan_processed$compound) 

plot(iwillsurvive_insecticides)

In [ ]:
#to make a fancy survival curve with a risk table summarising data

ggsurvplot(iwillsurvive_insecticides, data = lifespan_processed, risk.table = TRUE)
graph1 <- ggsurvplot(iwillsurvive_CamK1, data = lifespan_processed)